# Concrete Crack Benchmark — Unified Pipeline

Runs **every candidate model through the identical measurement rig** and produces a single comparison table (metric rows, one column per model):

- one stratified 70/15/15 split (seed=42), computed **once** and shared by all models;
- the same training loop (cross-entropy + Adam, best-validation checkpoint);
- the same latency protocol (batch=1, warm-up, `torch.cuda.synchronize`, median + p95);
- the same size / MACs / parameters / peak-GPU-memory measurements.

Because the rig is byte-identical across models, any difference in the final table is attributable to the **models themselves**, not to measurement differences.

**Usage.** Run top to bottom on an otherwise idle machine (latency fairness). Set `QUICK_TEST = True` in the configuration cell for a tiny-subset smoke run (~3–5 min) that verifies the pipeline end-to-end — its numbers are *not* meaningful. The full run takes ~25–35 minutes on an RTX 3070 Ti laptop GPU.

In [1]:
# === 1. Imports, seed, configuration ===================================
import os, time, copy, random, tempfile, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import (
    mobilenet_v2, MobileNet_V2_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
    mobilenet_v3_large, MobileNet_V3_Large_Weights,
    squeezenet1_1, SqueezeNet1_1_Weights,
    shufflenet_v2_x1_0, ShuffleNet_V2_X1_0_Weights,
)
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)
from thop import profile

# --- experiment configuration (identical to the per-model notebooks) ---
SEED        = 42
DATA_DIR    = r".\data"
IMG_SIZE    = 224
BATCH_SIZE  = 64
EPOCHS      = 3
LR          = 1e-4
NUM_WORKERS = 4
QUICK_TEST  = False    # True = tiny-subset smoke run (verifies the pipeline, ~3-5 min)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
if QUICK_TEST:
    print("QUICK MODE: tiny subset, 1 epoch - numbers are NOT meaningful")

d:\VsCode-projects\imageml-benchmark\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda | NVIDIA GeForce RTX 3070 Ti Laptop GPU


## 2. Model registry — the only model-specific code

Each entry supplies (a) a builder that returns the architecture with an ImageNet-pretrained backbone and a fresh 2-class head, and (b) the input normalization its pretrained weights expect. Five models use the ImageNet statistics; the TensorFlow-ported EfficientNet-Lite0 weights were trained with mean 0.5 / std 0.5 (`timm` default configuration for this model), so the pipeline uses those — the guiding principle is identical: *inputs must match the distribution the pretrained weights were trained on*.

Everything below this cell is the shared measurement rig and must stay identical for every model.

In [2]:
# === 2. Model registry =================================================
IMAGENET_NORM = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
HALF_NORM     = ((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))   # TF-ported EfficientNet-Lite weights

def build_mobilenet_v2():
    m = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    return m

def build_mobilenet_v3_small():
    m = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 2)
    return m

def build_mobilenet_v3_large():
    m = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1)
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 2)
    return m

def build_squeezenet():
    m = squeezenet1_1(weights=SqueezeNet1_1_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Conv2d(512, 2, kernel_size=1)   # fully-convolutional head
    m.num_classes = 2
    return m

def build_shufflenet():
    m = shufflenet_v2_x1_0(weights=ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m

def build_efficientnet_lite0():
    # num_classes=2 swaps the final classifier for a fresh Linear(1280, 2)
    return timm.create_model("tf_efficientnet_lite0", pretrained=True, num_classes=2)

MODELS = [
    ("MobileNetV2",        build_mobilenet_v2,        IMAGENET_NORM),
    ("MobileNetV3-Small",  build_mobilenet_v3_small,  IMAGENET_NORM),
    ("MobileNetV3-Large",  build_mobilenet_v3_large,  IMAGENET_NORM),
    ("SqueezeNet1.1",      build_squeezenet,          IMAGENET_NORM),
    ("ShuffleNetV2-x1.0",  build_shufflenet,          IMAGENET_NORM),
    ("EfficientNet-Lite0", build_efficientnet_lite0,  HALF_NORM),
]
print("Registered models:", [name for name, _, _ in MODELS])

Registered models: ['MobileNetV2', 'MobileNetV3-Small', 'MobileNetV3-Large', 'SqueezeNet1.1', 'ShuffleNetV2-x1.0', 'EfficientNet-Lite0']


## 3. Shared measurement rig

The functions below reproduce, verbatim, the methodology of the per-model notebooks:

- **Split** — computed once from the class labels; every model trains and is evaluated on the *same* image indices. In quick mode a small stratified subsample is drawn first.
- **Training** — cross-entropy + Adam at a low learning rate; after each epoch the model is validated, and the best-validation weights are restored before any measurement. The test set plays no role in model selection.
- **Latency** — batch=1, untimed warm-up passes, `torch.cuda.synchronize()` around every timed pass, median of the timed runs (p95 reported alongside). Eager-mode FP32 PyTorch.
- **Static metrics** — weights saved to disk for size (decimal MB); MACs via the `thop` profiler; parameters counted directly (thop misses some layer types, e.g. `timm`'s `BatchNormAct2d`); peak GPU memory for a single forward pass measured after clearing training-time allocations.

The seed is re-fixed before *each* model so every candidate sees the same shuffle order and augmentation draws.

In [3]:
# === 3. Shared measurement rig =========================================
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def make_split(targets, quick):
    # one stratified 70/15/15 split, shared by every model
    idx = np.arange(len(targets))
    if quick:                          # tiny stratified subsample for the smoke run
        rng = np.random.RandomState(SEED)
        idx = np.sort(np.concatenate([
            rng.choice(np.where(targets == c)[0], 300, replace=False)
            for c in np.unique(targets)
        ]))
    tr, tmp = train_test_split(idx, test_size=0.30,
                               stratify=targets[idx], random_state=SEED)
    va, te = train_test_split(tmp, test_size=0.50,
                              stratify=targets[tmp], random_state=SEED)
    return tr, va, te

_VIEW_CACHE = {}
def folder_views(norm):
    # (train_view, eval_view) of DATA_DIR for a given normalization; cached
    if norm not in _VIEW_CACHE:
        mean, std = norm
        eval_tf = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        train_tf = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(),          # the only augmentation
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        _VIEW_CACHE[norm] = (datasets.ImageFolder(DATA_DIR, transform=train_tf),
                             datasets.ImageFolder(DATA_DIR, transform=eval_tf))
    return _VIEW_CACHE[norm]

def make_loaders(norm, tr, va, te):
    train_view, eval_view = folder_views(norm)
    mk = lambda ds, sh: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=sh,
                                   num_workers=NUM_WORKERS, pin_memory=True,
                                   persistent_workers=(NUM_WORKERS > 0))
    return (mk(Subset(train_view, tr), True),
            mk(Subset(eval_view, va), False),
            mk(Subset(eval_view, te), False))

def train(model, train_loader, val_loader, epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    def run_epoch(loader, training):
        model.train() if training else model.eval()
        total, correct, loss_sum = 0, 0, 0.0
        with torch.set_grad_enabled(training):
            for x, y in loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                if training:
                    optimizer.zero_grad()
                out = model(x)
                loss = criterion(out, y)
                if training:
                    loss.backward(); optimizer.step()
                loss_sum += loss.item() * x.size(0)
                correct  += (out.argmax(1) == y).sum().item()
                total    += x.size(0)
        return loss_sum / total, correct / total

    best_acc, best_state = 0.0, None
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(train_loader, True)
        va_loss, va_acc = run_epoch(val_loader, False)
        print(f"    epoch {ep}: train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"val loss {va_loss:.4f} acc {va_acc:.4f}")
        if va_acc > best_acc:
            best_acc, best_state = va_acc, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)          # best-validation weights
    return best_acc

@torch.no_grad()
def evaluate(model, loader, pos_label):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        ps.append(model(x.to(device)).argmax(1).cpu().numpy())
        ys.append(y.numpy())
    y_true, y_pred = np.concatenate(ys), np.concatenate(ps)
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=pos_label)
    cm = confusion_matrix(y_true, y_pred)
    return acc, prec, rec, f1, cm

def measure_latency(model, device_, n_warmup, n_runs):
    m = copy.deepcopy(model).to(device_).eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device_)
    is_cuda = device_.type == "cuda"
    with torch.no_grad():
        for _ in range(n_warmup):                 # warm-up, untimed
            _ = m(x)
        if is_cuda:
            torch.cuda.synchronize()
        times = []
        for _ in range(n_runs):
            if is_cuda:
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = m(x)
            if is_cuda:
                torch.cuda.synchronize()          # wait for actual completion
            times.append((time.perf_counter() - t0) * 1000.0)   # -> ms
    t = np.array(times)
    return {"median_ms": float(np.median(t)), "p95_ms": float(np.percentile(t, 95))}

def static_metrics(model, name):
    # on-disk size of the trained weights
    tmp = os.path.join(tempfile.gettempdir(), f"pipeline_{name}.pt")
    torch.save(model.state_dict(), tmp)
    size_mb = os.path.getsize(tmp) / 1e6

    # MACs via thop; parameters counted directly (thop misses some layer
    # types, e.g. timm's BatchNormAct2d)
    m_cpu = copy.deepcopy(model).cpu().eval()
    macs, _ = profile(m_cpu, inputs=(torch.randn(1, 3, IMG_SIZE, IMG_SIZE),),
                      verbose=False)
    params = sum(p.numel() for p in m_cpu.parameters())

    # peak GPU memory for a single forward pass (inference footprint only)
    peak_mb = None
    if device.type == "cuda":
        gc.collect()
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            _ = model(torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device))
        torch.cuda.synchronize()
        peak_mb = torch.cuda.max_memory_allocated() / 1e6
    return size_mb, macs / 1e9, params, peak_mb

print("Measurement rig ready.")

Measurement rig ready.


## 4. Run all models

One shared split, then each model in turn: train → test metrics → latency (GPU, CPU) → size / MACs / parameters / peak memory. GPU memory is released between models. This is the only long-running cell.

In [4]:
# === 4. Run every model through the rig ================================
epochs = 1 if QUICK_TEST else EPOCHS
n_warmup, n_runs = (5, 30) if QUICK_TEST else (20, 100)

base = datasets.ImageFolder(DATA_DIR)
targets = np.array(base.targets)
pos_label = base.class_to_idx["Positive"]
tr, va, te = make_split(targets, QUICK_TEST)
print(f"Split (shared by all models): train {len(tr)} | val {len(va)} | "
      f"test {len(te)}  (stratified, seed={SEED})")

results = {}
for name, builder, norm in MODELS:
    t0 = time.time()
    print(f"\n=== {name} ===")
    seed_everything(SEED)                       # same RNG state for every model
    train_loader, val_loader, test_loader = make_loaders(norm, tr, va, te)

    model = builder().to(device)
    best_val = train(model, train_loader, val_loader, epochs)
    acc, prec, rec, f1, cm = evaluate(model, test_loader, pos_label)

    lat_gpu = (measure_latency(model, torch.device("cuda"), n_warmup, n_runs)
               if device.type == "cuda" else None)
    lat_cpu = measure_latency(model, torch.device("cpu"), n_warmup, n_runs)
    size_mb, gmacs, params, peak_mb = static_metrics(model, name)

    results[name] = dict(acc=acc, prec=prec, rec=rec, f1=f1, cm=cm,
                         lat_gpu=lat_gpu, lat_cpu=lat_cpu, size_mb=size_mb,
                         gmacs=gmacs, params=params, peak_mb=peak_mb,
                         best_val=best_val)
    print(f"    test acc {acc:.4f} | GPU {lat_gpu['median_ms']:.2f} ms | "
          f"CPU {lat_cpu['median_ms']:.2f} ms | {size_mb:.2f} MB | "
          f"{gmacs:.3f} GMACs | done in {time.time() - t0:.0f}s")

    del model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("\nAll models finished.")

Split (shared by all models): train 28000 | val 6000 | test 6000  (stratified, seed=42)

=== MobileNetV2 ===
    epoch 1: train loss 0.0147 acc 0.9958 | val loss 0.0035 acc 0.9983
    epoch 2: train loss 0.0037 acc 0.9989 | val loss 0.0018 acc 0.9993
    epoch 3: train loss 0.0013 acc 0.9995 | val loss 0.0034 acc 0.9993
    test acc 0.9993 | GPU 21.48 ms | CPU 69.26 ms | 9.15 MB | 0.326 GMACs | done in 398s

=== MobileNetV3-Small ===
    epoch 1: train loss 0.0296 acc 0.9914 | val loss 0.0026 acc 0.9992
    epoch 2: train loss 0.0035 acc 0.9992 | val loss 0.0016 acc 0.9993
    epoch 3: train loss 0.0027 acc 0.9994 | val loss 0.0011 acc 0.9997
    test acc 0.9990 | GPU 21.44 ms | CPU 39.03 ms | 6.22 MB | 0.061 GMACs | done in 227s

=== MobileNetV3-Large ===
    epoch 1: train loss 0.0195 acc 0.9941 | val loss 0.0015 acc 0.9997
    epoch 2: train loss 0.0031 acc 0.9991 | val loss 0.0027 acc 0.9992
    epoch 3: train loss 0.0014 acc 0.9996 | val loss 0.0025 acc 0.9993
    test acc 0.9992 

## 5. Comparison table

Metric rows in the requested order — **Accuracy, Inference Latency, Model Size, Computational Cost, Memory Usage** — followed by supplementary quality metrics; one column per model.

Conventions: latency is eager-mode FP32 PyTorch (no TensorRT / `torch.compile` / quantization); FLOPs = 2 × thop-measured MACs; model size is the `state_dict` on disk in decimal MB. mAP and Top-5 accuracy are not applicable to binary classification (no bounding-box annotations; top-5 is trivially 100% with two classes) — top-1 accuracy is reported instead.

In [5]:
# === 5. Comparison table (metric rows, one column per model) ===========
METRIC_ROWS = [
    "Accuracy (top-1)",
    "Inference Latency - GPU (batch=1, median)",
    "Inference Latency - GPU (batch=1, p95)",
    "Inference Latency - CPU (batch=1, median)",
    "Inference Latency - CPU (batch=1, p95)",
    "Model Size (weights on disk)",
    "Computational Cost (GMACs / GFLOPs)",
    "Memory Usage (peak GPU, single image)",
    "Precision (Positive = crack)",
    "Recall (Positive = crack)",
    "F1 score (Positive = crack)",
    "Parameters",
]

def format_column(r):
    fmt_ms = lambda d, k: f"{d[k]:.2f} ms" if d else "n/a"
    return [
        f"{r['acc'] * 100:.2f} %",
        fmt_ms(r["lat_gpu"], "median_ms"),
        fmt_ms(r["lat_gpu"], "p95_ms"),
        fmt_ms(r["lat_cpu"], "median_ms"),
        fmt_ms(r["lat_cpu"], "p95_ms"),
        f"{r['size_mb']:.2f} MB",
        f"{r['gmacs']:.3f} / {2 * r['gmacs']:.3f}",
        f"{r['peak_mb']:.1f} MB" if r["peak_mb"] is not None else "n/a",
        f"{r['prec']:.4f}",
        f"{r['rec']:.4f}",
        f"{r['f1']:.4f}",
        f"{r['params'] / 1e6:.2f} M",
    ]

comparison = pd.DataFrame({name: format_column(r) for name, r in results.items()},
                          index=METRIC_ROWS)
comparison.index.name = "Metric"

if QUICK_TEST:
    print("QUICK MODE - the numbers below are NOT meaningful\n")

# CSV export disabled to keep the project folder clean; uncomment to write
# comparison.to_csv(r".\benchmark_comparison.csv")

# presentation: metric names left-aligned, values right-aligned
(comparison.style
    .set_properties(**{"text-align": "right"})
    .set_table_styles([
        {"selector": "th.row_heading", "props": "text-align: left;"},
        {"selector": "th.col_heading", "props": "text-align: right;"},
        {"selector": "td, th", "props": "padding: 4px 12px;"},
    ]))

,MobileNetV2,MobileNetV3-Small,MobileNetV3-Large,SqueezeNet1.1,ShuffleNetV2-x1.0,EfficientNet-Lite0
Metric,,,,,,
Accuracy (top-1),99.93 %,99.90 %,99.92 %,99.88 %,99.92 %,99.82 %
"Inference Latency - GPU (batch=1, median)",21.48 ms,21.44 ms,23.11 ms,8.63 ms,6.89 ms,6.08 ms
"Inference Latency - GPU (batch=1, p95)",36.15 ms,36.20 ms,40.68 ms,13.82 ms,9.36 ms,7.16 ms
"Inference Latency - CPU (batch=1, median)",69.26 ms,39.03 ms,65.27 ms,39.47 ms,17.41 ms,20.76 ms
"Inference Latency - CPU (batch=1, p95)",93.69 ms,46.49 ms,87.28 ms,48.03 ms,27.89 ms,26.25 ms
Model Size (weights on disk),9.15 MB,6.22 MB,17.03 MB,2.92 MB,5.20 MB,13.77 MB
Computational Cost (GMACs / GFLOPs),0.326 / 0.652,0.061 / 0.123,0.234 / 0.467,0.263 / 0.526,0.152 / 0.303,0.366 / 0.733
"Memory Usage (peak GPU, single image)",47.3 MB,33.3 MB,61.4 MB,30.7 MB,32.1 MB,60.9 MB
Precision (Positive = crack),0.9993,0.9993,0.9990,0.9980,0.9990,0.9977


#### Conclusion:
The model selection depends on purposes and required accuracy. If the resource is not limited and you need **maximum accuracy**, **MobileNetV2** is the most relevant.

On the other hand if the error price is not high, then differences in accuracy between the models are not crucial. Thus, we need compare them by their production and memory costs, speed instead. ShuffleNetV2-x1.0 - it is the best performer here, it provides high speed , high accuracy, low memory consumption both on model and GPU VRAM despite higher GMACs / GFLOPs. ShuffleNetV2-x1.0 is 3.1 times faster that it nearest competitor on lightweight-class MobileNetV3-Small.

Of course, each model should be tested in production. However, if I have my own choice, I would definitely select **ShuffleNetV2-x1.0** for this task, as **it a real favourite in that benchmark**.